# LeetCode 3985 — Palindromic Subarray Sum bằng Python

- Link: https://leetcode.com/problems/palindromic-subarray-sum/
- Độ khó: **Hard**
- Chủ đề: Mảng con đối xứng (palindrome), Tổng tiền tố (prefix sum), Thuật toán Manacher, Hash + chặt nhị phân

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def getSum(self, nums: List[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho mảng `nums` gồm các số **nguyên dương**. Tìm **tổng lớn nhất** của một **mảng con liên tiếp** (subarray) là **palindrome** (đọc xuôi ngược như nhau). Mảng con một phần tử luôn là palindrome nên đáp án `≥ max(nums)`.

Có hai ý then chốt:

1. Vì mọi số đều **dương**, trong các palindrome cùng **tâm**, palindrome **dài nhất** có tổng lớn nhất (mỗi lần mở rộng thêm hai phần tử dương). Vậy chỉ cần xét **palindrome cực đại ở mỗi tâm** — `2n − 1` ứng viên.
2. Bán kính palindrome cực đại ở mọi tâm tính được trong `O(n)` bằng **thuật toán Manacher**; tổng của đoạn lấy từ **tổng tiền tố** trong `O(1)`.

Sau notebook này bạn sẽ:

1. Dùng tổng tiền tố để tính tổng đoạn trong `O(1)` (và tránh lỗi lệch chỉ số).
2. Viết brute force `O(n³)` và mở rộng quanh tâm `O(n²)`.
3. Hiểu và cài đặt **Manacher** (hai mảng `d1`, `d2` cho độ dài lẻ/chẵn), có trace.
4. Viết cách **hash + chặt nhị phân** `O(n log n)` làm đối chứng độc lập.
5. Đối chiếu các cách bằng test ngẫu nhiên có seed, đo thời gian ở `n = 10^5`, và xử lý biến thể có số **âm** bằng sparse table.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `nums`. Trong mọi mảng con liên tiếp `nums[l..r]` là palindrome (`nums[l+t] == nums[r−t]` với mọi `t`), tìm **tổng** lớn nhất.

**Ví dụ 1:** `[10,10]` → `20` (cả mảng là palindrome).

**Ví dụ 2:** `[1,2,3,2,1,5,6]` → `9`: mảng con `[1,2,3,2,1]`.

**Ví dụ 3:** `[7,1,2,1,7,3,4,3,4]` → `18`: mảng con `[7,1,2,1,7]` có tổng `18`.

**Ví dụ 4:** `[1,2,3,4,5]` → `5`: không có palindrome dài hơn 1; lớn nhất là phần tử `5`.

**Ví dụ 5:** `[1000]` → `1000`.

**Ràng buộc**

- `1 <= nums.length <= 10^5`
- `1 <= nums[i] <= 10^9`

In [1]:
from typing import List

EXAMPLES = [
    ([10, 10], 20),
    ([1, 2, 3, 2, 1, 5, 6], 9),
    ([7, 1, 2, 1, 7, 3, 4, 3, 4], 18),
    ([1, 2, 3, 4, 5], 5),
    ([1000], 1000),
]
for nums, expected in EXAMPLES:
    print(f"nums={nums} -> kỳ vọng {expected}")

nums=[10, 10] -> kỳ vọng 20
nums=[1, 2, 3, 2, 1, 5, 6] -> kỳ vọng 9
nums=[7, 1, 2, 1, 7, 3, 4, 3, 4] -> kỳ vọng 18
nums=[1, 2, 3, 4, 5] -> kỳ vọng 5
nums=[1000] -> kỳ vọng 1000


## 3. Kiến thức nền

### 3.1. Palindrome và kiểm tra bằng slicing

Một dãy là palindrome nếu bằng chính nó khi đảo ngược. Với list: `a == a[::-1]`. `a[::-1]` là bản đảo ngược (slicing bước `-1`).

In [2]:
print([1, 2, 1] == [1, 2, 1][::-1], [1, 2, 3] == [1, 2, 3][::-1], [5] == [5][::-1], [4, 4] == [4, 4][::-1])
# mảng con liên tiếp nums[l..r] (cả hai đầu) là nums[l:r+1]
nums = [1, 2, 3, 2, 1, 5, 6]
print(nums[0:5], nums[0:5] == nums[0:5][::-1])

True False True True
[1, 2, 3, 2, 1] True


### 3.2. Tổng tiền tố (prefix sum)

`P[0] = 0`, `P[i+1] = P[i] + nums[i]`. Khi đó tổng `nums[l..r]` (cả hai đầu) là `P[r+1] − P[l]`. **Chỉ số lệch 1** là chỗ dễ sai nhất: `P` dài `n + 1`.

In [3]:
from itertools import accumulate
nums = [1, 2, 3, 2, 1, 5, 6]
P = [0] + list(accumulate(nums))
print(P)                      # [0, 1, 3, 6, 8, 9, 14, 20]
print(P[5] - P[0], sum(nums[0:5]))      # tổng nums[0..4] = 9 cả hai cách
print(P[3] - P[3 - 1], nums[2])          # tổng đoạn một phần tử nums[2..2] = nums[2]

[0, 1, 3, 6, 8, 9, 14, 20]
9 9
3 3


### 3.3. Tâm của palindrome: lẻ và chẵn

Palindrome có độ dài **lẻ** có tâm là một phần tử `i`: `[i−k+1 .. i+k−1]` với bán kính `k ≥ 1` (độ dài `2k − 1`). Palindrome có độ dài **chẵn** có tâm ở giữa hai phần tử: `[i−k .. i+k−1]` với `k ≥ 1`, độ dài `2k` (tâm nằm giữa `i−1` và `i`). Tổng cộng `2n − 1` tâm (n tâm lẻ, n − 1 tâm chẵn có k ≥ 1).

In [4]:
nums = [1, 2, 1, 1, 2]
n = len(nums)
for i in range(n):
    k = 1
    while i - k >= 0 and i + k < n and nums[i - k] == nums[i + k]:
        k += 1
    print(f"tâm lẻ {i}: bán kính cực đại {k} -> [{i - k + 1}..{i + k - 1}] = {nums[i - k + 1:i + k]}")
for i in range(1, n):
    k = 0
    while i - k - 1 >= 0 and i + k < n and nums[i - k - 1] == nums[i + k]:
        k += 1
    if k:
        print(f"tâm chẵn trước {i}: bán kính {k} -> [{i - k}..{i + k - 1}] = {nums[i - k:i + k]}")

tâm lẻ 0: bán kính cực đại 1 -> [0..0] = [1]
tâm lẻ 1: bán kính cực đại 2 -> [0..2] = [1, 2, 1]
tâm lẻ 2: bán kính cực đại 1 -> [2..2] = [1]
tâm lẻ 3: bán kính cực đại 1 -> [3..3] = [1]
tâm lẻ 4: bán kính cực đại 1 -> [4..4] = [2]
tâm chẵn trước 3: bán kính 2 -> [1..4] = [2, 1, 1, 2]


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 3: `[7,1,2,1,7,3,4,3,4]`. Các tâm lẻ cho palindrome cực đại: tâm `2` (giá trị `2`): `1 2 1`, rồi `7 1 2 1 7` (bán kính 3), dừng vì ngoài đầu mảng. Tổng `7+1+2+1+7 = 18`. Tâm `6` (giá trị `4`): `3 4 3`, rồi `7 3 4 3 4`? `7 ≠ 4` dừng: tổng `10`. Tâm `7` (giá trị `3`): `4 3 4` tổng 11. Lớn nhất `18`.

**Câu hỏi:** có cần xét cả các palindrome *ngắn hơn* ở cùng tâm không? Không: ví dụ `1 2 1` (tổng 4) bị `7 1 2 1 7` (tổng 18) vượt. Vì mọi số dương, mở rộng chỉ làm tổng **tăng**. Nên mỗi tâm chỉ cần **palindrome cực đại**.

Cũng cần lưu ý: **palindrome dài nhất toàn mảng chưa chắc có tổng lớn nhất** — ví dụ `[1, 2, 1, 100, 100]`: palindrome dài nhất là `[1,2,1]` (tổng 4), nhưng `[100,100]` có tổng 200. Ta phải xét **mọi tâm**.

In [5]:
nums = [1, 2, 1, 100, 100]
P = [0] + list(accumulate(nums))
best_by_center = []
n = len(nums)
for i in range(n):
    k = 1
    while i - k >= 0 and i + k < n and nums[i - k] == nums[i + k]:
        k += 1
    best_by_center.append(("lẻ", i, k, P[i + k] - P[i - k + 1]))
for i in range(1, n):
    k = 0
    while i - k - 1 >= 0 and i + k < n and nums[i - k - 1] == nums[i + k]:
        k += 1
    if k:
        best_by_center.append(("chẵn", i, k, P[i + k] - P[i - k]))
for x in best_by_center:
    print(x)
print("lớn nhất:", max(x[3] for x in best_by_center))

('lẻ', 0, 1, 1)
('lẻ', 1, 2, 4)
('lẻ', 2, 1, 1)
('lẻ', 3, 1, 100)
('lẻ', 4, 1, 100)
('chẵn', 4, 1, 200)
lớn nhất: 200


## 5. Cách 1 — Brute Force: thử mọi mảng con

**Ý tưởng.** Với mỗi cặp `(l, r)`, kiểm tra `nums[l..r]` có phải palindrome không (so sánh với bản đảo), nếu đúng thì lấy tổng. Đây là bản dịch nguyên văn của đề, dùng làm trọng tài.

In [6]:
def brute_force(nums):
    n = len(nums)
    best = 0
    for l in range(n):
        for r in range(l, n):
            seg = nums[l:r + 1]
            if seg == seg[::-1]:
                best = max(best, sum(seg))
    return best

for nums, expected in EXAMPLES:
    print(nums, "->", brute_force(nums), "(kỳ vọng", expected, ")")

[10, 10] -> 20 (kỳ vọng 20 )
[1, 2, 3, 2, 1, 5, 6] -> 9 (kỳ vọng 9 )
[7, 1, 2, 1, 7, 3, 4, 3, 4] -> 18 (kỳ vọng 18 )
[1, 2, 3, 4, 5] -> 5 (kỳ vọng 5 )
[1000] -> 1000 (kỳ vọng 1000 )


**Độ phức tạp.** `O(n²)` mảng con, mỗi cái `O(n)` để so sánh và cộng: `O(n³)`. Chỉ cho `n ≤ 60`. **Nút thắt:** mỗi mảng con được kiểm tra từ đầu. Palindrome có cấu trúc: nếu `nums[l..r]` là palindrome thì `nums[l+1..r−1]` cũng vậy — nên kiểm tra theo **tâm** và mở rộng.

## 6. Bước đột phá — mỗi tâm chỉ xét palindrome cực đại

**Hai quan sát.**

1. *(Tối ưu theo tâm)* Mọi `nums[i] > 0`. Với một tâm cố định, palindrome bán kính `k+1` có thêm hai phần tử dương so với bán kính `k` ⇒ tổng lớn hơn. Vậy với mỗi tâm chỉ cần bán kính **lớn nhất**.
2. *(Tổng bằng tổng tiền tố)* Tổng của palindrome cực đại lấy ngay từ `P` trong `O(1)`.

Vậy đáp án = `max` trên `2n − 1` tâm của `P[hi+1] − P[lo]`, với `[lo..hi]` là palindrome cực đại tại tâm đó. Ta cần **bán kính cực đại của mọi tâm**: mở rộng trực tiếp tốn `O(n²)` trong trường hợp xấu (mảng toàn số bằng nhau), còn Manacher cho `O(n)`.

## 7. Các cách tiếp theo

- **Cách 2:** mở rộng quanh tâm `O(n²)`.
- **Cách 3:** **Manacher** `O(n)` — đáp án nộp.
- **Cách 4:** **hash + chặt nhị phân** `O(n log n)` (độc lập với Manacher).

### 7.1. Cách 2 — Mở rộng quanh tâm

**Giải thích từng đoạn.**

1. Tính `P`.
2. Với mỗi tâm lẻ `i`, `k = 1`; trong khi `i−k ≥ 0`, `i+k < n` và `nums[i−k] == nums[i+k]` thì `k += 1`. Palindrome cực đại: `[i−k+1, i+k−1]`, tổng `P[i+k] − P[i−k+1]`.
3. Với mỗi tâm chẵn giữa `i−1` và `i`: `k = 0`; trong khi `nums[i−k−1] == nums[i+k]` thì `k += 1`. Nếu `k ≥ 1`, tổng `P[i+k] − P[i−k]`.
4. Đáp án là `max`.

In [7]:
def expand_centers(nums):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    best = max(nums)
    for i in range(n):
        k = 1                                                   # lẻ: bán kính k, độ dài 2k-1
        while i - k >= 0 and i + k < n and nums[i - k] == nums[i + k]:
            k += 1
        best = max(best, P[i + k] - P[i - k + 1])
        k = 0                                                   # chẵn: tâm giữa i-1 và i
        while i - k - 1 >= 0 and i + k < n and nums[i - k - 1] == nums[i + k]:
            k += 1
        if k:
            best = max(best, P[i + k] - P[i - k])
    return best

for nums, expected in EXAMPLES:
    assert expand_centers(nums) == expected
print("Cách 2 khớp 5 ví dụ")

Cách 2 khớp 5 ví dụ


**Độ phức tạp.** `O(n²)` xấu nhất (mảng toàn số giống nhau: mỗi tâm mở rộng ~`n/2` lần), `O(n)` bộ nhớ. Ta đo ở mục 11.

### 7.2. Cách 3 — Manacher `O(n)` (đáp án nộp)

**Ý tưởng.** Không mở rộng từ đầu cho mỗi tâm; **tận dụng palindrome đã biết**. Giữ đoạn `[l, r]` là palindrome xa nhất bên phải đã gặp. Với tâm mới `i` nằm trong `[l, r]`, điểm đối xứng của `i` qua tâm của `[l, r]` là `j = l + r − i`. Bán kính tại `i` **ít nhất** bằng `min(bán kính tại j, r − i + 1)` (vì đoạn đối xứng). Chỉ cần mở rộng tiếp từ đó; nếu mở rộng vượt `r` thì cập nhật `[l, r]`.

- `d1[i]` = số palindrome **lẻ** tâm `i` (cũng là bán kính, vì mỗi bán kính `1..d1[i]` là một palindrome).
- `d2[i]` = số palindrome **chẵn** có tâm giữa `i−1` và `i`.

Mỗi lần mở rộng làm `r` tăng nên tổng số bước mở rộng `O(n)`.

In [8]:
def manacher(a):
    n = len(a)
    d1 = [0] * n                                  # lẻ
    l, r = 0, -1
    for i in range(n):
        k = 1 if i > r else min(d1[l + r - i], r - i + 1)
        while i - k >= 0 and i + k < n and a[i - k] == a[i + k]:
            k += 1
        d1[i] = k
        if i + k - 1 > r:
            l, r = i - k + 1, i + k - 1
    d2 = [0] * n                                  # chẵn
    l, r = 0, -1
    for i in range(n):
        k = 0 if i > r else min(d2[l + r - i + 1], r - i + 1)
        while i - k - 1 >= 0 and i + k < n and a[i - k - 1] == a[i + k]:
            k += 1
        d2[i] = k
        if i + k - 1 > r:
            l, r = i - k, i + k - 1
    return d1, d2

def best_manacher(nums):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    d1, d2 = manacher(nums)
    best = 0
    for i in range(n):
        k = d1[i]
        best = max(best, P[i + k] - P[i - k + 1])
        k = d2[i]
        if k:
            best = max(best, P[i + k] - P[i - k])
    return best

for nums, expected in EXAMPLES:
    assert best_manacher(nums) == expected
print("Cách 3 khớp 5 ví dụ")
print(manacher([1, 2, 1, 1, 2]))

Cách 3 khớp 5 ví dụ
([1, 2, 1, 1, 1], [0, 0, 0, 2, 0])


**Độ phức tạp.** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.3. Cách 4 — Hash + chặt nhị phân

**Ý tưởng.** Với tâm cố định, nếu bán kính `k` là palindrome thì mọi bán kính nhỏ hơn cũng vậy (**đơn điệu**) nên chặt nhị phân tìm bán kính lớn nhất. Kiểm tra "đoạn `nums[l..r]` là palindrome" trong `O(1)` bằng **hash đa thức**: so hash của đoạn đọc xuôi với hash của đoạn đọc ngược (hash trên mảng đảo). Dùng modulo lớn `2^61 − 1` và cơ số ngẫu nhiên để tránh va chạm.

In [9]:
import random

class PalHash:
    MOD = (1 << 61) - 1

    def __init__(self, a, seed=2026):
        n = len(a)
        self.n = n
        base = random.Random(seed).randrange(10 ** 6, self.MOD - 1)
        MOD = self.MOD
        self.pw = [1] * (n + 1)
        self.h = [0] * (n + 1)          # hash xuôi
        self.rh = [0] * (n + 1)         # hash của mảng đảo
        for i in range(n):
            self.pw[i + 1] = self.pw[i] * base % MOD
            self.h[i + 1] = (self.h[i] * base + a[i]) % MOD
            self.rh[i + 1] = (self.rh[i] * base + a[n - 1 - i]) % MOD

    def is_pal(self, l, r):             # nums[l..r] (cả hai đầu) có là palindrome không
        MOD = self.MOD
        ln = r - l + 1
        fwd = (self.h[r + 1] - self.h[l] * self.pw[ln]) % MOD
        a, b = self.n - 1 - r, self.n - 1 - l          # chỉ số tương ứng trong mảng đảo
        bwd = (self.rh[b + 1] - self.rh[a] * self.pw[ln]) % MOD
        return fwd == bwd

def best_hash(nums):
    n = len(nums)
    P = [0] * (n + 1)
    for i, x in enumerate(nums):
        P[i + 1] = P[i] + x
    H = PalHash(nums)
    best = 0
    for i in range(n):
        lo, hi = 1, min(i + 1, n - i)             # lẻ: bán kính k, đoạn [i-k+1, i+k-1]
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if H.is_pal(i - mid + 1, i + mid - 1):
                lo = mid
            else:
                hi = mid - 1
        best = max(best, P[i + lo] - P[i - lo + 1])
        lo, hi = 0, min(i, n - i)                 # chẵn: tâm giữa i-1 và i, đoạn [i-k, i+k-1]
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if H.is_pal(i - mid, i + mid - 1):
                lo = mid
            else:
                hi = mid - 1
        if lo:
            best = max(best, P[i + lo] - P[i - lo])
    return best

for nums, expected in EXAMPLES:
    assert best_hash(nums) == expected
print("Cách 4 khớp 5 ví dụ")

Cách 4 khớp 5 ví dụ


**Độ phức tạp.** `O(n log n)` thời gian (mỗi tâm `~17` lần kiểm tra `O(1)`), `O(n)` bộ nhớ. Đúng với xác suất cao (va chạm ≈ `n log n / 2^61`); khác với Manacher là xác định tuyệt đối. Ta dùng làm đối chứng độc lập.

## 8. TRACE từng bước

Ta theo dõi Manacher (phần lẻ) trên `[1, 2, 1, 2, 1, 3, 1, 2, 1]`: ở mỗi `i` in cửa sổ `[l, r]`, giá trị lấy từ gương `j = l + r − i`, và số lần mở rộng thêm.

| cột | ý nghĩa |
|---|---|
| `[l, r]` | palindrome xa nhất bên phải đã gặp |
| `j` | điểm đối xứng của `i` qua tâm `[l, r]` |
| `k0` | bán kính khởi đầu từ gương: `min(d1[j], r−i+1)` hoặc `1` |
| `k` | bán kính cuối `d1[i]` sau khi mở rộng |

In [10]:
def trace_manacher_odd(a):
    n = len(a)
    print("a =", a)
    d1 = [0] * n
    l, r = 0, -1
    for i in range(n):
        if i > r:
            k = 1
            note = "ngoài cửa sổ -> bắt đầu k=1"
        else:
            j = l + r - i
            k = min(d1[j], r - i + 1)
            note = f"trong cửa sổ [{l},{r}], gương j={j}, d1[j]={d1[j]}, r-i+1={r - i + 1} -> k0={k}"
        k0 = k
        while i - k >= 0 and i + k < n and a[i - k] == a[i + k]:
            k += 1
        d1[i] = k
        moved = ""
        if i + k - 1 > r:
            l, r = i - k + 1, i + k - 1
            moved = f"  => cửa sổ mới [{l},{r}]"
        print(f"i={i}: {note}; mở rộng {k - k0} lần -> d1[{i}]={k}{moved}")
    print("d1 =", d1)
    return d1

trace_manacher_odd([1, 2, 1, 2, 1, 3, 1, 2, 1])

a = [1, 2, 1, 2, 1, 3, 1, 2, 1]
i=0: ngoài cửa sổ -> bắt đầu k=1; mở rộng 0 lần -> d1[0]=1  => cửa sổ mới [0,0]
i=1: ngoài cửa sổ -> bắt đầu k=1; mở rộng 1 lần -> d1[1]=2  => cửa sổ mới [0,2]
i=2: trong cửa sổ [0,2], gương j=0, d1[j]=1, r-i+1=1 -> k0=1; mở rộng 2 lần -> d1[2]=3  => cửa sổ mới [0,4]
i=3: trong cửa sổ [0,4], gương j=1, d1[j]=2, r-i+1=2 -> k0=2; mở rộng 0 lần -> d1[3]=2
i=4: trong cửa sổ [0,4], gương j=0, d1[j]=1, r-i+1=1 -> k0=1; mở rộng 0 lần -> d1[4]=1
i=5: ngoài cửa sổ -> bắt đầu k=1; mở rộng 3 lần -> d1[5]=4  => cửa sổ mới [2,8]
i=6: trong cửa sổ [2,8], gương j=4, d1[j]=1, r-i+1=3 -> k0=1; mở rộng 0 lần -> d1[6]=1
i=7: trong cửa sổ [2,8], gương j=3, d1[j]=2, r-i+1=2 -> k0=2; mở rộng 0 lần -> d1[7]=2
i=8: trong cửa sổ [2,8], gương j=2, d1[j]=3, r-i+1=1 -> k0=1; mở rộng 0 lần -> d1[8]=1
d1 = [1, 2, 3, 2, 1, 4, 1, 2, 1]


[1, 2, 3, 2, 1, 4, 1, 2, 1]

Quan sát: ở `i = 3` (giá trị `2`) cửa sổ `[0..4]` cho gương `j = 1` với `d1[1] = 2`, nhưng bị chặn bởi `r − i + 1 = 2` nên bắt đầu `k = 2` mà không phải so lại các phần tử đã biết. Thêm trace tính đáp án cho ví dụ 3 (Manacher + tổng tiền tố):

In [11]:
def trace_answer(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    d1, d2 = manacher(nums)
    print("nums =", nums)
    print("P    =", P)
    print("d1   =", d1)
    print("d2   =", d2)
    best = 0
    for i in range(n):
        k = d1[i]
        s = P[i + k] - P[i - k + 1]
        print(f"  tâm lẻ {i}: [{i - k + 1}..{i + k - 1}] tổng {s}")
        best = max(best, s)
        k = d2[i]
        if k:
            s = P[i + k] - P[i - k]
            print(f"  tâm chẵn trước {i}: [{i - k}..{i + k - 1}] tổng {s}")
            best = max(best, s)
    print("đáp án:", best)
    return best

trace_answer([7, 1, 2, 1, 7, 3, 4, 3, 4])

nums = [7, 1, 2, 1, 7, 3, 4, 3, 4]
P    = [0, 7, 8, 10, 11, 18, 21, 25, 28, 32]
d1   = [1, 1, 3, 1, 1, 1, 2, 2, 1]
d2   = [0, 0, 0, 0, 0, 0, 0, 0, 0]
  tâm lẻ 0: [0..0] tổng 7
  tâm lẻ 1: [1..1] tổng 1
  tâm lẻ 2: [0..4] tổng 18
  tâm lẻ 3: [3..3] tổng 1
  tâm lẻ 4: [4..4] tổng 7
  tâm lẻ 5: [5..5] tổng 3
  tâm lẻ 6: [5..7] tổng 10
  tâm lẻ 7: [6..8] tổng 11
  tâm lẻ 8: [8..8] tổng 4
đáp án: 18


18

## 9. Code nộp LeetCode

In [12]:
from typing import List

class Solution:
    def getSum(self, nums: List[int]) -> int:
        n = len(nums)
        P = [0] * (n + 1)                          # tổng tiền tố: tổng nums[l..r] = P[r+1] - P[l]
        for i, x in enumerate(nums):
            P[i + 1] = P[i] + x

        best = 0
        d1 = [0] * n                               # d1[i]: số palindrome lẻ tâm i (bán kính)
        l, r = 0, -1
        for i in range(n):
            k = 1 if i > r else min(d1[l + r - i], r - i + 1)
            while i - k >= 0 and i + k < n and nums[i - k] == nums[i + k]:
                k += 1
            d1[i] = k
            if i + k - 1 > r:
                l, r = i - k + 1, i + k - 1
            best = max(best, P[i + k] - P[i - k + 1])   # palindrome lẻ cực đại tâm i

        d2 = [0] * n                               # d2[i]: số palindrome chẵn tâm giữa i-1 và i
        l, r = 0, -1
        for i in range(n):
            k = 0 if i > r else min(d2[l + r - i + 1], r - i + 1)
            while i - k - 1 >= 0 and i + k < n and nums[i - k - 1] == nums[i + k]:
                k += 1
            d2[i] = k
            if i + k - 1 > r:
                l, r = i - k, i + k - 1
            if k:
                best = max(best, P[i + k] - P[i - k])   # palindrome chẵn cực đại
        return best

for nums, expected in EXAMPLES:
    assert Solution().getSum(nums) == expected
print("Code nộp khớp 5 ví dụ")

Code nộp khớp 5 ví dụ


## 10. Vì sao thuật toán đúng

**(a) Đưa về palindrome cực đại ở mỗi tâm.** Mỗi palindrome `nums[l..r]` có một tâm duy nhất (lẻ hoặc chẵn) và bán kính `k`. Với cùng tâm, các palindrome bán kính `1, 2, ..., K` (`K` là bán kính cực đại) **lồng nhau**: palindrome bán kính `k+1` = palindrome bán kính `k` cộng hai phần tử hai đầu. Vì phần tử dương, tổng tăng nghiêm ngặt theo `k`. Do đó, trong số các palindrome cùng tâm, palindrome bán kính `K` có tổng lớn nhất. Đáp án = `max` trên mọi tâm của tổng palindrome cực đại.

**(b) Tổng tiền tố.** Tổng `nums[l..r] = P[r+1] − P[l]`. Với tâm lẻ `i` bán kính `k`: `l = i − k + 1`, `r = i + k − 1` ⇒ `P[i+k] − P[i−k+1]`. Với tâm chẵn giữa `i−1` và `i` bán kính `k`: `l = i − k`, `r = i + k − 1` ⇒ `P[i+k] − P[i−k]`.

**(c) Manacher.** Giữ `[l, r]` là palindrome có `r` lớn nhất trong các palindrome cực đại đã tính. Với `i ≤ r`, gương `j = l + r − i` có palindrome cực đại bán kính `d1[j]`; vì `[l, r]` đối xứng quanh tâm của nó, palindrome tâm `j` bên trong `[l, r]` phản chiếu thành palindrome tâm `i`, nên bán kính tại `i` `≥ min(d1[j], r − i + 1)`. Chỉ phần vượt `r` mới cần so sánh thật.

**(d) Độ phức tạp `O(n)`.** Vòng `while` chỉ chạy khi `i + k` vượt `r` (hoặc ngay từ `i > r`), và mỗi lần chạy làm `r` tăng. `r` chỉ tăng tối đa `n` lần nên tổng số lần mở rộng `O(n)` (cộng `n` lần `min`).

**(e) Biên.** Mọi tâm lẻ có `d1[i] ≥ 1` nên `best ≥ max(nums)`. Tâm chẵn có thể có `d2[i] = 0` (không có palindrome chẵn) nên bỏ qua. Giá trị lớn tới `10^9 × 10^5 = 10^14` an toàn vì Python dùng số nguyên lớn tuỳ ý.

## 11. Phân tích Time/Space

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| Brute force | `O(n³)` | `O(n)` |
| Mở rộng quanh tâm | `O(n²)` xấu nhất | `O(n)` |
| Manacher (nộp) | `O(n)` | `O(n)` |
| Hash + chặt nhị phân | `O(n log n)` (xác suất đúng cao) | `O(n)` |

Đo trên dữ liệu xấu nhất cho mở rộng quanh tâm — mảng toàn số giống nhau:

In [13]:
import time

def timed(fn, nums):
    t0 = time.time()
    r = fn(nums)
    return r, round(time.time() - t0, 3)

for n in (500, 1000, 2000, 4000):
    nums = [7] * n
    print(f"n={n:5d} toàn số 7: mở rộng {timed(expand_centers, nums)}, Manacher {timed(best_manacher, nums)}")

rng = random.Random(1)
big_equal = [10 ** 9] * 100000
big_rand = [rng.randint(1, 10 ** 9) for _ in range(100000)]
big_small = [rng.randint(1, 2) for _ in range(100000)]
big_period = [1, 2, 3] * 33333
print("n=1e5, Manacher:")
for name, arr in [("toàn 1e9", big_equal), ("ngẫu nhiên lớn", big_rand), ("nhị phân {1,2}", big_small), ("tuần hoàn 1,2,3", big_period)]:
    print(f"  {name:16s}", timed(Solution().getSum, arr))
print("n=1e5 hash + chặt nhị phân (nhị phân {1,2}):", timed(best_hash, big_small))

n=  500 toàn số 7: mở rộng (3500, 0.009), Manacher (3500, 0.0)
n= 1000 toàn số 7: mở rộng (7000, 0.038), Manacher (7000, 0.001)


n= 2000 toàn số 7: mở rộng (14000, 0.173), Manacher (14000, 0.002)


n= 4000 toàn số 7: mở rộng (28000, 0.769), Manacher (28000, 0.003)
n=1e5, Manacher:
  toàn 1e9         (100000000000000, 0.074)
  ngẫu nhiên lớn   (999984194, 0.05)


  nhị phân {1,2}   (51, 0.068)
  tuần hoàn 1,2,3  (3, 0.042)


n=1e5 hash + chặt nhị phân (nhị phân {1,2}): (51, 1.734)


Thời gian mở rộng quanh tâm tăng khoảng **4 lần** mỗi khi `n` gấp đôi (bình phương), còn Manacher chỉ tăng gấp đôi (tuyến tính). Với `n = 10^5` mở rộng quanh tâm trên mảng toàn số giống nhau sẽ cần cỡ `10^10 / 2` bước — không thể trong giới hạn thời gian.

## 12. Bộ test

Ví dụ đề, edge case, phản ví dụ, đối chiếu ngẫu nhiên có seed: vét cạn ↔ mở rộng ↔ Manacher ↔ hash ↔ code nộp.

In [14]:
S = Solution()
for nums, expected in EXAMPLES:
    for fn in (brute_force, expand_centers, best_manacher, best_hash, S.getSum):
        assert fn(nums) == expected, (fn.__name__, nums)
print("Ví dụ đề: mọi cách khớp")

Ví dụ đề: mọi cách khớp


In [15]:
# Edge case
assert S.getSum([5]) == 5
assert S.getSum([3, 3]) == 6
assert S.getSum([1, 2]) == 2                         # không có palindrome dài; lấy phần tử lớn nhất
assert S.getSum([4, 1, 4]) == 9
assert S.getSum([1, 100, 1]) == 102
assert S.getSum([1, 2, 1, 100, 100]) == 200          # palindrome dài nhất [1,2,1] KHÔNG cho tổng lớn nhất
assert S.getSum([5, 1, 1, 5, 9]) == 12               # chẵn: 5 1 1 5
assert S.getSum([2, 2, 2, 2, 2]) == 10               # cả mảng
assert S.getSum([1, 2, 3, 4, 3, 2, 1]) == 16         # cả mảng đối xứng
assert S.getSum([10 ** 9] * 100000) == 10 ** 9 * 100000   # giá trị lớn
print("Edge case: OK")

Edge case: OK


**Phản ví dụ** cho các cách nghĩ sai (xem mục 13): chỉ xét độ dài lẻ, lấy palindrome dài nhất, lệch chỉ số tiền tố, hash modulo nhỏ.

In [16]:
# 1) Đối chiếu ngẫu nhiên nhỏ với vét cạn
rng = random.Random(2026)
for it in range(1500):
    n = rng.randint(1, 30)
    hi = rng.choice([1, 2, 3, 10, 100])
    nums = [rng.randint(1, hi) for _ in range(n)]
    ref = brute_force(nums)
    assert expand_centers(nums) == ref, nums
    assert best_manacher(nums) == ref, nums
    assert best_hash(nums) == ref, nums
    assert S.getSum(nums) == ref, nums
print("1500 input ngẫu nhiên nhỏ: các cách khớp vét cạn")

# 2) Manacher ↔ mở rộng trên input vừa (nhiều palindrome)
for it in range(200):
    n = rng.randint(50, 400)
    nums = [rng.randint(1, rng.choice([1, 2, 3])) for _ in range(n)]
    assert S.getSum(nums) == expand_centers(nums) == best_hash(nums)
print("200 input vừa: Manacher ↔ mở rộng ↔ hash khớp")

# 3) Kiểm tra d1, d2 của Manacher với định nghĩa
def d_naive(a):
    n = len(a)
    d1, d2 = [0] * n, [0] * n
    for i in range(n):
        k = 1
        while i - k >= 0 and i + k < n and a[i - k] == a[i + k]:
            k += 1
        d1[i] = k
        k = 0
        while i - k - 1 >= 0 and i + k < n and a[i - k - 1] == a[i + k]:
            k += 1
        d2[i] = k
    return d1, d2
for it in range(500):
    a = [rng.randint(1, 2) for _ in range(rng.randint(1, 40))]
    assert manacher(a) == d_naive(a), a
print("manacher(a) = (d1, d2) đúng định nghĩa trên 500 mảng")

1500 input ngẫu nhiên nhỏ: các cách khớp vét cạn


200 input vừa: Manacher ↔ mở rộng ↔ hash khớp
manacher(a) = (d1, d2) đúng định nghĩa trên 500 mảng


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét palindrome độ dài lẻ

Palindrome độ dài chẵn (ví dụ `[10, 10]`) có tâm **giữa hai phần tử**; nếu chỉ mở rộng quanh từng phần tử bạn bỏ sót chúng.

In [17]:
def wrong_only_odd(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    best = 0
    for i in range(n):
        k = 1
        while i - k >= 0 and i + k < n and nums[i - k] == nums[i + k]:
            k += 1
        best = max(best, P[i + k] - P[i - k + 1])
    return best

print("sai :", wrong_only_odd([10, 10]))               # 10
print("đúng:", Solution().getSum([10, 10]))             # 20

sai : 10
đúng: 20


**Cách đúng:** thêm tâm chẵn (hoặc dùng cả `d1` và `d2`).

### Lỗi 2 — Lấy palindrome **dài nhất** thay vì tổng lớn nhất

Palindrome dài nhất có thể gồm các số nhỏ, trong khi một palindrome ngắn hơn gồm các số lớn cho tổng cao hơn.

In [18]:
def wrong_longest(nums):
    n = len(nums)
    best_len, best_sum = 0, 0
    for l in range(n):
        for r in range(l, n):
            seg = nums[l:r + 1]
            if seg == seg[::-1] and (len(seg) > best_len):
                best_len, best_sum = len(seg), sum(seg)
    return best_sum

nums = [1, 2, 1, 100, 100]
print("sai :", wrong_longest(nums))               # 4
print("đúng:", Solution().getSum(nums))            # 200

sai : 4
đúng: 200


**Cách đúng:** xét **mọi** tâm và lấy `max` của tổng, không phải của độ dài.

### Lỗi 3 — Lệch chỉ số tổng tiền tố

Tổng `nums[l..r]` là `P[r+1] − P[l]`. Viết `P[r] − P[l]` hoặc `P[r+1] − P[l−1]` là lỗi kinh điển. Với tâm lẻ: `P[i+k] − P[i−k+1]` (không phải `P[i−k]`).

In [19]:
def wrong_prefix(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    best = 0
    for i in range(n):
        k = 1
        while i - k >= 0 and i + k < n and nums[i - k] == nums[i + k]:
            k += 1
        best = max(best, P[i + k] - P[i - k])          # SAI: thừa nums[i-k]
        j = 0
        while i - j - 1 >= 0 and i + j < n and nums[i - j - 1] == nums[i + j]:
            j += 1
        if j:
            best = max(best, P[i + j] - P[i - j])
    return best

nums = [7, 1, 2, 1, 7, 3, 4, 3, 4]
print("sai :", wrong_prefix(nums))                      # lệch: cộng thừa một phần tử (và P[-1] ở biên là chỉ số âm của Python!)
print("đúng:", Solution().getSum(nums))                  # 18

sai : 17
đúng: 18


**Cách đúng:** kiểm tra bằng ví dụ nhỏ: tâm lẻ bán kính `1` phải cho tổng `nums[i]`, tức `P[i+1] − P[i]`.

### Lỗi 4 — Hash với modulo nhỏ (va chạm)

Hash đa thức với modulo nhỏ sinh va chạm: hai đoạn **không** đối xứng có hash xuôi bằng hash ngược ⇒ tưởng palindrome. Dùng modulo `2^61 − 1` (hoặc hai modulo) và cơ số ngẫu nhiên.

In [20]:
def best_hash_weak(nums, MOD=101, base=31):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    h = [0] * (n + 1); rh = [0] * (n + 1); pw = [1] * (n + 1)
    for i in range(n):
        pw[i + 1] = pw[i] * base % MOD
        h[i + 1] = (h[i] * base + nums[i]) % MOD
        rh[i + 1] = (rh[i] * base + nums[n - 1 - i]) % MOD
    def is_pal(l, r):
        ln = r - l + 1
        f = (h[r + 1] - h[l] * pw[ln]) % MOD
        a, b = n - 1 - r, n - 1 - l
        g = (rh[b + 1] - rh[a] * pw[ln]) % MOD
        return f == g
    best = 0
    for i in range(n):
        lo, hi = 1, min(i + 1, n - i)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if is_pal(i - mid + 1, i + mid - 1): lo = mid
            else: hi = mid - 1
        best = max(best, P[i + lo] - P[i - lo + 1])
    return best

rng = random.Random(8)
for _ in range(3000):
    nums = [rng.randint(1, 9) for _ in range(rng.randint(5, 14))]
    if best_hash_weak(nums) > wrong_only_odd(nums):
        print("nums =", nums)
        print("hash yếu cho:", best_hash_weak(nums), "| đáp án (chỉ lẻ) đúng:", wrong_only_odd(nums))
        break
else:
    print("(không tìm thấy va chạm trong 3000 lần thử)")

nums = [1, 1, 6, 9, 5, 6, 7, 1, 3, 1, 2]
hash yếu cho: 39 | đáp án (chỉ lẻ) đúng: 9


**Cách đúng:** modulo lớn (`2^61 − 1`), cơ số ngẫu nhiên; hoặc tốt nhất dùng Manacher (xác định, không hash).

## 14. Mẹo & nhận dạng pattern

- **"Palindrome liên tiếp (substring/subarray)"** → nghĩ ngay tới **Manacher** (`O(n)`) hoặc mở rộng quanh tâm; hash + chặt nhị phân là phương án thay thế.
- **Tâm lẻ và tâm chẵn** là hai trường hợp riêng: `d1` và `d2`, hoặc chèn ký tự phân cách `#` để gộp.
- **Phần tử dương** ⇒ mở rộng chỉ làm tổng tăng ⇒ chỉ cần palindrome cực đại ở mỗi tâm. Nếu có số âm/không dương, phải xét mọi bán kính (xem biến thể).
- **Tổng tiền tố** với `P` dài `n + 1`: tổng `[l..r] = P[r+1] − P[l]`.
- **Hash palindrome**: so hash xuôi và hash của mảng đảo; modulo lớn + cơ số ngẫu nhiên.
- Mở rộng quanh tâm tốt cho `n ≤ 5000`; với `n = 10^5` và dữ liệu xấu (toàn số bằng nhau) cần Manacher.
- Luôn thử `[x, x]` và `[1, 2, 1, 100, 100]` để bắt các lỗi chẵn/lẻ và "dài nhất ≠ tổng lớn nhất".

## 15. Biến thể

1. **Cho phép số âm / không dương:** palindrome cực đại không còn tối ưu; phải xét mọi bán kính `k ≤ d1[i]`. Với tâm lẻ `i`, tổng bán kính `k` là `nums[i] + 2·(P[i+k] − P[i+1])`, nên cần `max P[i+k]` với `k ∈ [1, d1[i]]` — truy vấn max đoạn trên `P` (sparse table).
2. **Đếm số mảng con là palindrome:** `Σ d1[i] + Σ d2[i]`.
3. **Trả chỉ số `(l, r)`** của palindrome tốt nhất.
4. **Palindrome theo giá trị** (gương theo `|a|` hoặc `a mod m`): đổi phép so sánh.

Ta cài biến thể 1 và 2 và kiểm chứng bằng vét cạn.

In [21]:
def best_with_negatives(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    d1, d2 = manacher(nums)
    # sparse table cho max trên P
    sp = [P[:]]
    j = 1
    while (1 << j) <= len(P):
        prev, step = sp[-1], 1 << (j - 1)
        sp.append([max(prev[i], prev[i + step]) for i in range(len(prev) - step)])
        j += 1
    def rmq(lo, hi):                                  # max P[lo..hi]
        j = (hi - lo + 1).bit_length() - 1
        return max(sp[j][lo], sp[j][hi - (1 << j) + 1])
    best = max(nums)
    for i in range(n):
        K_ = d1[i]
        # tổng bán kính k: nums[i] + 2*(P[i+k] - P[i+1]); k = 1..K_
        best = max(best, nums[i] + 2 * (rmq(i + 1, i + K_) - P[i + 1]))
        K_ = d2[i]
        if K_:
            # chẵn bán kính k: 2*(P[i+k] - P[i]); k = 1..K_
            best = max(best, 2 * (rmq(i + 1, i + K_) - P[i]))
    return best

def brute_any(nums):
    n = len(nums)
    best = None
    for l in range(n):
        for r in range(l, n):
            seg = nums[l:r + 1]
            if seg == seg[::-1]:
                s = sum(seg)
                best = s if best is None else max(best, s)
    return best

def count_pal_subarrays(nums):
    d1, d2 = manacher(nums)
    return sum(d1) + sum(d2)

def count_brute(nums):
    n = len(nums)
    return sum(1 for l in range(n) for r in range(l, n) if nums[l:r + 1] == nums[l:r + 1][::-1])

rng = random.Random(15)
for _ in range(600):
    nums = [rng.randint(-3, 3) for _ in range(rng.randint(1, 25))]
    assert best_with_negatives(nums) == brute_any(nums), nums
    pos = [abs(x) + 1 for x in nums]
    assert count_pal_subarrays(pos) == count_brute(pos)
print("Biến thể có số âm (sparse table) và đếm palindrome: khớp vét cạn")
print(best_with_negatives([-5, 3, -5, -1]), best_with_negatives([2, -1, 2]))

Biến thể có số âm (sparse table) và đếm palindrome: khớp vét cạn
3 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi mảng con, so sánh với bản đảo | `O(n³)` | `O(n)` | trọng tài, `n ≤ 60` |
| 2. Mở rộng quanh tâm | mỗi tâm mở rộng tối đa | `O(n²)` | `O(n)` | `n ≤ 5000`, đơn giản |
| 3. Manacher (nộp) | tận dụng palindrome đã biết | `O(n)` | `O(n)` | luôn dùng cho đề (`n = 10^5`) |
| 4. Hash + nhị phân | `is_pal` `O(1)` bằng hash, chặt nhị phân bán kính | `O(n log n)` | `O(n)` | đối chứng độc lập; cần modulo lớn |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `P = [0] + list(accumulate([1, 2, 3]))` là gì? Tổng `nums[1..2]` tính bằng `P` là bao nhiêu?

**Bài 2 (tính tay).** `nums = [3, 1, 3, 1, 3]`. Liệt kê các tâm lẻ và bán kính cực đại; đáp án là bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thiện `grow(nums, l, r)` mở rộng `[l, r]` ra hai phía khi `nums[l-1] == nums[r+1]` và trả `(l, r)` cuối cùng.

**Bài 4 (tự viết lại).** Viết `my_solve(nums)` bằng mở rộng quanh tâm với `grow` và tổng tiền tố.

**Bài 5 (biến thể / mini challenge).** Viết `count_palindromes(nums)` đếm số mảng con là palindrome bằng Manacher (`Σ d1 + Σ d2`), và kiểm chứng bằng vét cạn.

In [22]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def grow(nums, l, r):
    # TODO: while l - 1 >= 0 and r + 1 < len(nums) and nums[l - 1] == nums[r + 1]: l -= 1; r += 1
    # trả về (l, r)
    pass

# Bài 4 — TODO
def my_solve(nums):
    # TODO: P = tổng tiền tố; với mỗi i: grow(i, i) và grow(i, i+1) nếu nums[i] == nums[i+1]
    pass

# Bài 5 — TODO
def count_palindromes(nums):
    # TODO: d1, d2 = manacher(nums); return sum(d1) + sum(d2)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `P[0] = 0`, mỗi bước cộng thêm một phần tử.
- Mức 2: `P = [0, 1, 3, 6]`.
- Mức 3: `nums[1..2]` có tổng `P[3] − P[1] = 6 − 1 = 5`.

**Bài 2.**
- Mức 1: tâm `0` có bán kính `1` (đứng ở đầu mảng); tâm `1` (giá trị `1`): `3 1 3`, rồi hết chỗ mở rộng phía trái? `nums[-1]` không tồn tại nên dừng ở bán kính 2.
- Mức 2: tâm `2` (giá trị `3`): `1 3 1`, rồi `3 1 3 1 3` (bán kính 3).
- Mức 3: cả mảng là palindrome, tổng `11`.

**Bài 3.**
- Mức 1: vòng `while` kiểm tra biên và so sánh hai phần tử ngoài cùng.
- Mức 2: mỗi lần thành công `l -= 1`, `r += 1`.
- Mức 3: `return l, r`.

**Bài 4.**
- Mức 1: tâm lẻ bắt đầu `(i, i)`.
- Mức 2: tâm chẵn bắt đầu `(i, i+1)` nếu `nums[i] == nums[i+1]`.
- Mức 3: tổng `P[r+1] - P[l]`.

**Bài 5.**
- Mức 1: `d1[i]` palindrome lẻ tâm `i`.
- Mức 2: `d2[i]` palindrome chẵn tâm giữa `i-1` và `i`.
- Mức 3: `sum(d1) + sum(d2)`.

## 19. Đáp án tham khảo

In [23]:
# Bài 1
P = [0] + list(accumulate([1, 2, 3]))
assert P == [0, 1, 3, 6] and P[3] - P[1] == 5

# Bài 2
assert Solution().getSum([3, 1, 3, 1, 3]) == 11
d1, d2 = manacher([3, 1, 3, 1, 3])
assert d1 == [1, 2, 3, 2, 1] and d2 == [0, 0, 0, 0, 0]

# Bài 3
def grow_ans(nums, l, r):
    while l - 1 >= 0 and r + 1 < len(nums) and nums[l - 1] == nums[r + 1]:
        l -= 1
        r += 1
    return l, r

# Bài 4
def my_solve_ans(nums):
    n = len(nums)
    P = [0] + list(accumulate(nums))
    best = 0
    for i in range(n):
        l, r = grow_ans(nums, i, i)
        best = max(best, P[r + 1] - P[l])
        if i + 1 < n and nums[i] == nums[i + 1]:
            l, r = grow_ans(nums, i, i + 1)
            best = max(best, P[r + 1] - P[l])
    return best

rng = random.Random(33)
for _ in range(400):
    nums = [rng.randint(1, 3) for _ in range(rng.randint(1, 30))]
    assert my_solve_ans(nums) == Solution().getSum(nums) == brute_force(nums)
print("Bài 1–4 OK")

Bài 1–4 OK


In [24]:
# Bài 5
def count_palindromes_ans(nums):
    d1, d2 = manacher(nums)
    return sum(d1) + sum(d2)

assert count_palindromes_ans([1, 2, 1]) == 4         # 1,2,1 và 1 2 1
assert count_palindromes_ans([5, 5, 5]) == 6         # 3 đơn + 2 cặp + 1 ba
rng = random.Random(44)
for _ in range(300):
    nums = [rng.randint(1, 3) for _ in range(rng.randint(1, 30))]
    assert count_palindromes_ans(nums) == count_brute(nums)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template Manacher (lẻ + chẵn) với tổng tiền tố:**

```python
P = [0] * (n + 1)
for i, x in enumerate(nums): P[i + 1] = P[i] + x

d1 = [0] * n; l, r = 0, -1                         # lẻ
for i in range(n):
    k = 1 if i > r else min(d1[l + r - i], r - i + 1)
    while i - k >= 0 and i + k < n and nums[i - k] == nums[i + k]: k += 1
    d1[i] = k
    if i + k - 1 > r: l, r = i - k + 1, i + k - 1
    best = max(best, P[i + k] - P[i - k + 1])

d2 = [0] * n; l, r = 0, -1                         # chẵn: tâm giữa i-1 và i
for i in range(n):
    k = 0 if i > r else min(d2[l + r - i + 1], r - i + 1)
    while i - k - 1 >= 0 and i + k < n and nums[i - k - 1] == nums[i + k]: k += 1
    d2[i] = k
    if i + k - 1 > r: l, r = i - k, i + k - 1
    if k: best = max(best, P[i + k] - P[i - k])
```

**Checklist trước khi submit:**

- [ ] Cả tâm **lẻ** và tâm **chẵn**.
- [ ] Phần tử dương ⇒ chỉ cần palindrome cực đại ở mỗi tâm; xét **mọi** tâm (không chỉ palindrome dài nhất).
- [ ] Tổng tiền tố: lẻ `P[i+k] − P[i−k+1]`, chẵn `P[i+k] − P[i−k]`.
- [ ] Manacher: gương `j = l + r − i` (lẻ) và `l + r − i + 1` (chẵn); cập nhật `[l, r]` khi vượt `r`.
- [ ] Không dùng hash modulo nhỏ; ưu tiên Manacher.
- [ ] Đã đối chiếu vét cạn trên input nhỏ, và thử `[10,10]`, `[1,2,1,100,100]`.